# Phase 3 one-QRC two-head readout

This is the non-hybrid version of the operational idea.

It uses one RF-QRC ring reservoir and the same raw quantum feature vector, but trains two readout heads:

```text
ridge regression head   -> continuous future volatility
logistic classifier head -> q80/q90/q95 event probability
```

The question is whether the crisis signal is present in the shared reservoir features but diluted by the regression objective.


In [ ]:
from __future__ import annotations

from pathlib import Path
import os
import subprocess
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 220)

def find_repo_root() -> Path:
    cwd = Path.cwd().resolve()
    for p in [cwd, *cwd.parents]:
        if p.name == 'qpitome-qrc-volatility' and (p / 'scripts').exists() and (p / 'src').exists():
            return p
    raise RuntimeError('Open this notebook from inside qpitome-qrc-volatility.')

ROOT = find_repo_root()
os.chdir(ROOT)
TABLES = ROOT / 'results' / 'tables'
FIGURES = ROOT / 'results' / 'figures'
TABLES.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)
print('Repo root:', ROOT)


## Run one-QRC two-head script

In [ ]:
script = ROOT / 'scripts' / 'run_phase3_one_qrc_two_head_readout.py'
if not script.exists():
    raise FileNotFoundError(script)

cmd = [
    sys.executable, str(script),
    '--leak', '0.3',
    '--input-scale', str(np.pi / 3),
    '--random-scale', '0.35',
    '--ridge-alpha', '3000',
    '--logistic-C', '1.0',
]
print(' '.join(cmd))
subprocess.run(cmd, cwd=ROOT, check=True)


## Load outputs

In [ ]:
reg_path = TABLES / 'phase3_one_qrc_two_head_regression_metrics.csv'
clf_path = TABLES / 'phase3_one_qrc_two_head_classifier_metrics.csv'
pred_path = TABLES / 'phase3_one_qrc_two_head_predictions.csv'
summary_path = TABLES / 'phase3_one_qrc_two_head_test_summary.csv'

reg = pd.read_csv(reg_path)
clf = pd.read_csv(clf_path)
preds = pd.read_csv(pred_path)
summary = pd.read_csv(summary_path)

display(reg[reg['split'].eq('test')])
display(clf[clf['split'].eq('test')].sort_values('event'))
display(preds.head())


## Compare with existing main RF-QRC / Phase 2 / ESN table if available

In [ ]:
context_path = TABLES / 'phase3_clean_raw_forecast_comparison.csv'
if context_path.exists():
    context = pd.read_csv(context_path)
    display(context)
else:
    context = pd.DataFrame()
    print('No context table found:', context_path)

test_reg = reg[reg['split'].eq('test')].copy()
test_clf = clf[clf['split'].eq('test')].copy()

compact_reg = test_reg[[
    'head', 'rmse', 'qlike', 'corr', 'actual_std', 'pred_std',
    'q80_f1', 'q90_f1', 'q95_f1', 'top20_pred_actual_ratio', 'effective_rank_train'
]]
compact_clf = test_clf[[
    'head', 'event', 'precision', 'recall', 'f1', 'average_precision',
    'roc_auc', 'called_rate', 'event_rate', 'decision_threshold'
]]
display(compact_reg)
display(compact_clf.sort_values('event'))


## Figures

In [ ]:
def savefig(path: Path):
    plt.tight_layout()
    plt.savefig(path, dpi=180, bbox_inches='tight')
    print('Saved:', path)
    plt.show()

# Classifier metrics by event tier.
plot = test_clf.set_index('event').loc[['q80', 'q90', 'q95']]
plot[['precision', 'recall', 'f1', 'average_precision']].plot(kind='bar', figsize=(9, 5))
plt.ylabel('score')
plt.title('One-QRC two-head classifier performance')
plt.xticks(rotation=0)
savefig(FIGURES / 'phase3_one_qrc_two_head_classifier_metrics.png')

# Regression forecast trace.
test = preds[preds['split'].astype(str).str.lower().eq('test')].copy()
x = pd.to_datetime(test['date'], errors='coerce')
if x.isna().all():
    x = np.arange(len(test))
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(x, test['actual_future_rv_20d'], label='actual', linewidth=2)
ax.plot(x, test['rf_qrc_ring_ridge_pred'], label='ridge regression head')
ax.set_title('One-QRC two-head: regression head forecast')
ax.set_ylabel('future RV 20d')
ax.legend()
savefig(FIGURES / 'phase3_one_qrc_two_head_regression_forecast.png')

# q95 crisis probability trace.
fig, ax = plt.subplots(figsize=(14, 4.5))
ax.plot(x, test['rf_qrc_ring_q95_prob'], label='q95 crisis probability')
ax.fill_between(x, 0, 1, where=test['actual_q95_event'].astype(bool), alpha=0.15, label='actual q95 event')
ax.set_title('One-QRC two-head: q95 crisis probability')
ax.set_ylabel('probability')
ax.set_ylim(0, 1)
ax.legend()
savefig(FIGURES / 'phase3_one_qrc_two_head_q95_probability.png')

# Probability versus realized event label distributions.
fig, ax = plt.subplots(figsize=(7, 4.5))
event = test['actual_q95_event'].astype(bool)
ax.hist(test.loc[~event, 'rf_qrc_ring_q95_prob'], bins=30, alpha=0.6, label='non-event')
ax.hist(test.loc[event, 'rf_qrc_ring_q95_prob'], bins=30, alpha=0.6, label='q95 event')
ax.set_title('q95 classifier probability separation')
ax.set_xlabel('predicted q95 probability')
ax.set_ylabel('count')
ax.legend()
savefig(FIGURES / 'phase3_one_qrc_two_head_q95_probability_hist.png')


## Interpretation gate

In [ ]:
print('This is useful if the classifier q95 F1/precision improves over the regression-implied q95 signal, while the regression head remains the same one-reservoir forecast.')

reg_q95 = float(test_reg['q95_f1'].iloc[0])
clf_q95 = float(test_clf.loc[test_clf['event'].eq('q95'), 'f1'].iloc[0])
clf_q95_precision = float(test_clf.loc[test_clf['event'].eq('q95'), 'precision'].iloc[0])
clf_q95_recall = float(test_clf.loc[test_clf['event'].eq('q95'), 'recall'].iloc[0])

print(f'Regression-implied q95 F1: {reg_q95:.3f}')
print(f'Classifier q95 F1:         {clf_q95:.3f}')
print(f'Classifier q95 precision:  {clf_q95_precision:.3f}')
print(f'Classifier q95 recall:     {clf_q95_recall:.3f}')

if clf_q95 > reg_q95:
    print('The shared reservoir benefits from target-specific readout.')
else:
    print('The classifier head does not improve q95 F1 over the regression-implied signal in this configuration.')
